In [ ]:
"""Train a layer-wise GRU exit classifier on summary predictions only.

Install: pip install torch transformers datasets sentencepiece
Run: python train_summary_exit_gru.py --samples 100 --epochs 3

One GPU (or CPU); frozen Llama, teacher-forced reference highlights.
This trains an exit classifier, not an actual early-exit decoding engine.
"""

import argparse
from pathlib import Path

import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader

import itertools
import torch
import transformers
from datasets import load_dataset
from dataclasses import dataclass

local_model_path: str = "facebook/layerskip-llama2-7B"

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
dtype = torch.float16 if device.type == "cuda" else torch.float32

print(f"Using device: {device}")

tokenizer = transformers.AutoTokenizer.from_pretrained(local_model_path)

model = transformers.AutoModelForCausalLM.from_pretrained(
    local_model_path,
    use_safetensors=True,
    torch_dtype=dtype,
).to(device)





/home/markphan24/anaconda3/envs/layer_skip/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda:0


Loading checkpoint shards: 100%|██████████| 3/3 [-00:00<00:00, -11.75it/s]


In [4]:
def encode_example(tokenizer, article, highlight, max_length):
    """Format: <s>[Prompt: ]  {article} [Summary: ] {highlight}</s>"""
    encode = lambda text: tokenizer.encode(text, add_special_tokens=False)

    if tokenizer.bos_token_id is None or tokenizer.eos_token_id is None:
        raise ValueError("This example requires BOS and EOS tokens.")

    prefix = [tokenizer.bos_token_id] + encode("[Prompt: ]  ")
    separator = encode(" [Summary: ] ")
    target = encode(highlight) + [tokenizer.eos_token_id]

    # Truncate only the article; preserve the entire highlight and EOS.
    article_budget = max_length - len(prefix) - len(separator) - len(target)
    if article_budget < 1:
        raise ValueError("Highlight is too long; increase --max-length.")

    prompt = prefix + encode(article)[:article_budget] + separator
    input_ids = prompt + target

    # Shift manually: each position predicts the next token.
    # Only summary tokens and EOS contribute to the classifier's training.
    labels = [-100] * len(prompt) + target
    labels = labels[1:] + [-100]

    return {"input_ids": input_ids, "labels": labels}


In [ ]:
article = "The capital of France is"
highlight = "Paris"

result = encode_example(tokenizer, article, highlight, max_length=4096)

result


In [ ]:
def collate_examples(examples, pad_token_id):
    """Right padding. Attention and loss masks have different purposes."""
    length = max(len(x["input_ids"]) for x in examples)
    ids = torch.full((len(examples), length), pad_token_id, dtype=torch.long)
    labels = torch.full_like(ids, -100)
    attention = torch.zeros_like(ids)
    for i, example in enumerate(examples):
        n = len(example["input_ids"])
        ids[i, :n] = torch.tensor(example["input_ids"])
        labels[i, :n] = torch.tensor(example["labels"])
        attention[i, :n] = 1
    return {"input_ids": ids, "attention_mask": attention, "labels": labels}


class EarlyExitGRU(nn.Module):
    def __init__(self, llm_hidden_size=4096, gru_hidden_size=128):
        super().__init__()
        self.input_norm = nn.LayerNorm(llm_hidden_size)
        # The sequence dimension is decoder depth, NOT text positions.
        # Unidirectional: decision at layer l sees only layers 1,...,l.
        self.gru = nn.GRU(llm_hidden_size, gru_hidden_size, batch_first=True)
        self.exit_head = nn.Linear(gru_hidden_size, 1)

    def forward(self, hidden_states, recurrent_state=None):
        # hidden_states: [number_of_target_tokens, early_layers, hidden_size]
        outputs, state = self.gru(self.input_norm(hidden_states), recurrent_state)
        return self.exit_head(outputs).squeeze(-1), state

    @torch.no_grad()
    def step(self, layer_hidden, recurrent_state=None):
        """One layer of one decoding step; reset state for each new token."""
        logits, state = self(layer_hidden.unsqueeze(1).float(), recurrent_state)
        return logits[:, 0].sigmoid(), state


@torch.no_grad()
def extract_summary_features(model, batch, projection_chunk_size=32):
    """Return CPU features [N,L-1,H] and binary exit labels [N,L-1].

    For standard Hugging Face Llama:
      hidden_states[0] = embeddings
      hidden_states[1:L] = raw outputs of layers 1,...,L-1
      last_hidden_state = final layer output AFTER final RMSNorm
    Thus final states must NOT be normalized a second time.
    """
    model.eval()
    device = model.get_input_embeddings().weight.device
    ids = batch["input_ids"].to(device)
    attention = batch["attention_mask"].to(device)
    labels = batch["labels"].to(device)

    # State at position t predicts token at position t+1.
    # This selects the BOS state for the first highlight token.
    target_mask = labels[:, 1:].ne(-100)
    target_mask &= attention[:, :-1].bool() & attention[:, 1:].bool()
    rows, positions = target_mask.nonzero(as_tuple=True)
    if rows.numel() == 0:
        raise ValueError("Batch contains no summary targets.")

    outputs = model.model(
        input_ids=ids,
        attention_mask=attention,
        output_hidden_states=True,
        use_cache=False,
        return_dict=True,
    )
    num_layers = len(model.model.layers)
    if len(outputs.hidden_states) != num_layers + 1:
        raise ValueError("Unexpected Llama hidden-state layout.")

    def predict_tokens(states, normalize):
        # Project only selected summary-prediction positions, in chunks.
        predictions = []
        for start in range(0, rows.numel(), projection_chunk_size):
            stop = start + projection_chunk_size
            h = states[rows[start:stop], positions[start:stop]]
            if normalize:
                h = model.model.norm(h)
            predictions.append(model.lm_head(h).argmax(-1).cpu())
        return torch.cat(predictions)

    teacher_ids = predict_tokens(outputs.last_hidden_state, normalize=False)
    features, exit_targets = [], []
    for layer_index in range(1, num_layers):
        raw_hidden = outputs.hidden_states[layer_index]
        features.append(raw_hidden[rows, positions].cpu())
        early_ids = predict_tokens(raw_hidden, normalize=True)
        exit_targets.append(early_ids.eq(teacher_ids).float())

    # Llama stays frozen. No full [B,T,V] logits are stored for each layer.
    return torch.stack(features, dim=1), torch.stack(exit_targets, dim=1)


def train_epoch(model, classifier, loader, optimizer, token_batch_size=32):
    classifier.train()
    device = next(classifier.parameters()).device
    total_loss, count = 0.0, 0
    for batch in loader:
        features, targets = extract_summary_features(model, batch)
        order = torch.randperm(features.shape[0])
        for indices in order.split(token_batch_size):
            x = features[indices].to(device=device, dtype=torch.float32)
            y = targets[indices].to(device)
            optimizer.zero_grad(set_to_none=True)
            # GRU state resets here: each target token is an independent
            # sequence across Llama layers, with no cross-token recurrence.
            exit_logits, _ = classifier(x)
            loss = F.binary_cross_entropy_with_logits(exit_logits, y)
            loss.backward()
            nn.utils.clip_grad_norm_(classifier.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item() * y.numel()
            count += y.numel()
    return total_loss / max(count, 1)


def main():
    from functools import partial
    from datasets import load_dataset
    from transformers import AutoModelForCausalLM, AutoTokenizer

    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--model", default="facebook/layerskip-llama2-7B")
    parser.add_argument("--samples", type=int, default=100)
    parser.add_argument("--epochs", type=int, default=3)
    parser.add_argument("--batch-size", type=int, default=1)
    parser.add_argument("--token-batch-size", type=int, default=32)
    parser.add_argument("--max-length", type=int, default=1024)
    parser.add_argument("--gru-hidden-size", type=int, default=128)
    parser.add_argument("--lr", type=float, default=1e-3)
    parser.add_argument("--output", default="summary_exit_gru.pt")
    args = parser.parse_args()
    for name in ("samples", "epochs", "batch_size", "token_batch_size",
                 "max_length", "gru_hidden_size"):
        if getattr(args, name) <= 0:
            parser.error(f"{name} must be positive")
    torch.manual_seed(42)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    dtype = torch.float16 if device.type == "cuda" else torch.float32
    tokenizer = AutoTokenizer.from_pretrained(args.model)
    tokenizer.padding_side = "right"
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    # Deliberately single-device: features, RMSNorm, and LM head must agree.
    model = AutoModelForCausalLM.from_pretrained(
        args.model, torch_dtype=dtype, attn_implementation="sdpa",
    ).to(device)
    model.requires_grad_(False)
    model.eval()
    if args.max_length > model.config.max_position_embeddings:
        parser.error("max-length exceeds the model context limit")
    data = load_dataset(
        "abisee/cnn_dailymail", "3.0.0", split=f"train[:{args.samples}]"
    )
    examples = [encode_example(tokenizer, row["article"], row["highlights"],
                               args.max_length) for row in data]
    loader = DataLoader(
        examples, batch_size=args.batch_size, shuffle=True,
        collate_fn=partial(collate_examples, pad_token_id=tokenizer.pad_token_id),
    )
    classifier = EarlyExitGRU(model.config.hidden_size, args.gru_hidden_size).to(device)
    optimizer = torch.optim.AdamW(classifier.parameters(), lr=args.lr)
    for epoch in range(args.epochs):
        loss = train_epoch(model, classifier, loader, optimizer, args.token_batch_size)
        print(f"Epoch {epoch + 1}: summary-token exit BCE = {loss:.6f}", flush=True)
    destination = Path(args.output)
    destination.parent.mkdir(parents=True, exist_ok=True)
    torch.save({
        "state_dict": {k: v.detach().cpu() for k, v in classifier.state_dict().items()},
        "llm_hidden_size": model.config.hidden_size,
        "gru_hidden_size": args.gru_hidden_size,
        "num_early_layers": len(model.model.layers) - 1,
        "base_model": args.model,
        "label_definition": "intermediate argmax equals final-layer argmax",
        "training_args": vars(args),
    }, destination)
    print(f"Saved {destination}")


if __name__ == "__main__":
    main()
